# Gold Validation Report — PySpark on Amazon EMR

This notebook validates the six Gold outputs after the Silver-to-Gold notebook has written them.

It does not perform transformations or rewrite Gold data.


## 1. Imports and paths

In [1]:
from pyspark.sql import functions as F

GOLD_BASE_PATH = "s3://airline-dataset-2020-2025/Gold/"

PATHS = {
    "FACT_FLIGHTS": GOLD_BASE_PATH + "FACT_FLIGHTS/",
    "DIM_AIRLINE": GOLD_BASE_PATH + "DIM_AIRLINE/",
    "DIM_AIRPORT": GOLD_BASE_PATH + "DIM_AIRPORT/",
    "DIM_DATE": GOLD_BASE_PATH + "DIM_DATE/",
    "DIM_ROUTE": GOLD_BASE_PATH + "DIM_ROUTE/",
    "ML_DATASET": GOLD_BASE_PATH + "ML_DATASET/"
}


Starting Spark application


ID,YARN Application ID,Kind,State,Spark UI,Driver log,Current session?
0,application_1785032150392_0001,pyspark,idle,Link,Link,✔


SparkSession available as 'spark'.


## 2. Read Gold outputs

In [2]:
fact_df = spark.read.parquet(PATHS["FACT_FLIGHTS"])
dim_airline_df = spark.read.parquet(PATHS["DIM_AIRLINE"])
dim_airport_df = spark.read.parquet(PATHS["DIM_AIRPORT"])
dim_date_df = spark.read.parquet(PATHS["DIM_DATE"])
dim_route_df = spark.read.parquet(PATHS["DIM_ROUTE"])
ml_df = spark.read.parquet(PATHS["ML_DATASET"])

gold_tables = {
    "FACT_FLIGHTS": fact_df,
    "DIM_AIRLINE": dim_airline_df,
    "DIM_AIRPORT": dim_airport_df,
    "DIM_DATE": dim_date_df,
    "DIM_ROUTE": dim_route_df,
    "ML_DATASET": ml_df
}


## 3. Row and column counts

In [3]:
for name, df in gold_tables.items():
    print(name, "rows:", df.count(), "columns:", len(df.columns))


('FACT_FLIGHTS', 'rows:', 40910253, 'columns:', 55)
('DIM_AIRLINE', 'rows:', 25, 'columns:', 9)
('DIM_ROUTE', 'rows:', 8734, 'columns:', 11)
('ML_DATASET', 'rows:', 39895374, 'columns:', 31)
('DIM_AIRPORT', 'rows:', 391, 'columns:', 15)
('DIM_DATE', 'rows:', 2192, 'columns:', 10)

## 4. Fact missing-value percentages

In [4]:
fact_rows = fact_df.count()

fact_null_pct = fact_df.select([
    F.round(
        F.sum(F.when(F.col(c).isNull(), 1).otherwise(0))
        / F.lit(fact_rows) * 100,
        4
    ).alias(c)
    for c in fact_df.columns
])

fact_null_pct.show(truncate=False)


+---------+-------+-------------------+-------------------+----------------+--------------+--------+----------+-------------------------------+---------------------------------------+----------+----------+---------------------------+-------------+-----------+---------------+-------------+-----------------+----------------+---------------+--------+-------+-------+------+--------+--------+--------+--------+-----------+-----------+-------------------------+---------+--------+-------------------+------------+-------------+------------------------+----------------------+----------------+----------------------+----------------------+----------------------+---------------------+------------+------------+--------+-------------+-----------------+-----------------+-----------------------+--------------+-------------+-------------------+----+-----+
|FlightKey|DateKey|MarketingAirlineKey|OperatingAirlineKey|OriginAirportKey|DestAirportKey|RouteKey|FlightDate|Flight_Number_Marketing_Airline|Operate

## 5. FlightKey uniqueness and duplicate-record count

In [5]:
duplicate_flight_keys = (
    fact_df
    .groupBy("FlightKey")
    .count()
    .filter(F.col("count") > 1)
)

duplicate_summary = duplicate_flight_keys.agg(
    F.count("*").alias("DuplicatedKeyGroups"),
    F.coalesce(
        F.sum(F.col("count") - 1),
        F.lit(0)
    ).alias("DuplicateRecordCount")
)

duplicate_summary.show(truncate=False)


+-------------------+--------------------+
|DuplicatedKeyGroups|DuplicateRecordCount|
+-------------------+--------------------+
|0                  |0                   |
+-------------------+--------------------+

## 6. Dimension-key uniqueness

In [6]:
dimension_checks = [
    ("DIM_AIRLINE", dim_airline_df, "AirlineKey"),
    ("DIM_AIRPORT", dim_airport_df, "AirportKey"),
    ("DIM_DATE", dim_date_df, "DateKey"),
    ("DIM_ROUTE", dim_route_df, "RouteKey")
]

for name, df, key in dimension_checks:
    duplicates = (
        df.groupBy(key)
        .count()
        .filter(F.col("count") > 1)
        .count()
    )
    print(name, "duplicate key groups:", duplicates)


('DIM_AIRLINE', 'duplicate key groups:', 0)
('DIM_AIRPORT', 'duplicate key groups:', 0)
('DIM_DATE', 'duplicate key groups:', 0)
('DIM_ROUTE', 'duplicate key groups:', 0)

## 7. Foreign-key integrity

In [7]:
fk_report = fact_df.select(
    F.sum(F.when(F.col("DateKey").isNull(), 1).otherwise(0)).alias("MissingDateKey"),
    F.sum(F.when(F.col("MarketingAirlineKey").isNull(), 1).otherwise(0)).alias("MissingMarketingAirlineKey"),
    F.sum(F.when(F.col("OperatingAirlineKey").isNull(), 1).otherwise(0)).alias("MissingOperatingAirlineKey"),
    F.sum(F.when(F.col("OriginAirportKey").isNull(), 1).otherwise(0)).alias("MissingOriginAirportKey"),
    F.sum(F.when(F.col("DestAirportKey").isNull(), 1).otherwise(0)).alias("MissingDestAirportKey"),
    F.sum(F.when(F.col("RouteKey").isNull(), 1).otherwise(0)).alias("MissingRouteKey")
)

fk_report.show(truncate=False)


+--------------+--------------------------+--------------------------+-----------------------+---------------------+---------------+
|MissingDateKey|MissingMarketingAirlineKey|MissingOperatingAirlineKey|MissingOriginAirportKey|MissingDestAirportKey|MissingRouteKey|
+--------------+--------------------------+--------------------------+-----------------------+---------------------+---------------+
|0             |0                         |0                         |0                      |0                    |0              |
+--------------+--------------------------+--------------------------+-----------------------+---------------------+---------------+

## 8. Business-rule violation counts

In [8]:
business_rules = fact_df.select(
    F.sum(
        F.when(~F.col("Month").between(1, 12), 1).otherwise(0)
    ).alias("InvalidMonth"),
    F.sum(
        F.when(F.col("Distance") <= 0, 1).otherwise(0)
    ).alias("InvalidDistance"),
    F.sum(
        F.when(F.col("AirTime") < 0, 1).otherwise(0)
    ).alias("NegativeAirTime"),
    F.sum(
        F.when(F.col("TaxiOut") < 0, 1).otherwise(0)
    ).alias("NegativeTaxiOut"),
    F.sum(
        F.when(F.col("TaxiIn") < 0, 1).otherwise(0)
    ).alias("NegativeTaxiIn"),
    F.sum(
        F.when(
            (F.col("Cancelled") == 1)
            & (F.col("Diverted") == 1),
            1
        ).otherwise(0)
    ).alias("CancelledAndDiverted"),
    F.sum("ArrDelayNullExceptionFlag").alias("ArrDelayNullExceptions")
)

business_rules.show(truncate=False)


+------------+---------------+---------------+---------------+--------------+--------------------+----------------------+
|InvalidMonth|InvalidDistance|NegativeAirTime|NegativeTaxiOut|NegativeTaxiIn|CancelledAndDiverted|ArrDelayNullExceptions|
+------------+---------------+---------------+---------------+--------------+--------------------+----------------------+
|0           |0              |0              |0              |0             |0                   |5                     |
+------------+---------------+---------------+---------------+--------------+--------------------+----------------------+

## 9. Reliability-score ranges

In [9]:
dim_airline_df.select(
    F.min("ReliabilityScore").alias("MinAirlineScore"),
    F.max("ReliabilityScore").alias("MaxAirlineScore")
).show()

dim_airport_df.select(
    F.min("DepartureReliabilityScore").alias("MinDepartureScore"),
    F.max("DepartureReliabilityScore").alias("MaxDepartureScore"),
    F.min("ArrivalReliabilityScore").alias("MinArrivalScore"),
    F.max("ArrivalReliabilityScore").alias("MaxArrivalScore")
).show()

dim_route_df.select(
    F.min("ReliabilityScore").alias("MinRouteScore"),
    F.max("ReliabilityScore").alias("MaxRouteScore")
).show()


+---------------+---------------+
|MinAirlineScore|MaxAirlineScore|
+---------------+---------------+
|          81.05|          89.49|
+---------------+---------------+

+-----------------+-----------------+---------------+---------------+
|MinDepartureScore|MaxDepartureScore|MinArrivalScore|MaxArrivalScore|
+-----------------+-----------------+---------------+---------------+
|            65.38|            100.0|            0.0|          95.54|
+-----------------+-----------------+---------------+---------------+

+-------------+-------------+
|MinRouteScore|MaxRouteScore|
+-------------+-------------+
|         10.0|        100.0|
+-------------+-------------+

## 10. ML target and split distribution

In [10]:
ml_df.groupBy("DatasetSplit").count().orderBy("DatasetSplit").show()

ml_df.groupBy("DatasetSplit", "ArrDel15").count().orderBy(
    "DatasetSplit",
    "ArrDel15"
).show()


+------------+--------+
|DatasetSplit|   count|
+------------+--------+
|        Test| 7597495|
|       Train|24872650|
|  Validation| 7425229|
+------------+--------+

+------------+--------+--------+
|DatasetSplit|ArrDel15|   count|
+------------+--------+--------+
|        Test|       0| 5912161|
|        Test|       1| 1685334|
|       Train|       0|20444792|
|       Train|       1| 4427858|
|  Validation|       0| 5894149|
|  Validation|       1| 1531080|
+------------+--------+--------+

## 11. Year-month partition coverage

In [11]:
fact_df.groupBy("Year", "Month").count().orderBy(
    "Year",
    "Month"
).show(100, truncate=False)

ml_df.groupBy("Year", "Month").count().orderBy(
    "Year",
    "Month"
).show(100, truncate=False)


+----+-----+------+
|Year|Month|count |
+----+-----+------+
|2020|1    |660556|
|2020|2    |623103|
|2020|3    |701274|
|2020|4    |331238|
|2020|5    |192412|
|2020|6    |237264|
|2020|7    |370859|
|2020|8    |398470|
|2020|9    |345294|
|2020|10   |374538|
|2020|11   |389587|
|2020|12   |397802|
|2021|1    |379384|
|2021|2    |350170|
|2021|3    |467126|
|2021|4    |473936|
|2021|5    |520059|
|2021|6    |573779|
|2021|7    |615703|
|2021|8    |611494|
|2021|9    |567916|
|2021|10   |595373|
|2021|11   |576693|
|2021|12   |580238|
|2022|1    |563737|
|2022|2    |519952|
|2022|3    |590542|
|2022|4    |580290|
|2022|5    |602950|
|2022|6    |602057|
|2022|7    |618790|
|2022|8    |613649|
|2022|9    |580391|
|2022|10   |595322|
|2022|11   |567507|
|2022|12   |578321|
|2023|1    |573877|
|2023|2    |536229|
|2023|3    |616234|
|2023|4    |596676|
|2023|5    |616630|
|2023|6    |613577|
|2023|7    |638995|
|2023|8    |640236|
|2023|9    |604715|
|2023|10   |635538|
|2023|11   |599814|


# Validation Conclusion

Use the results above to confirm:

- Keys are complete and unique at their intended grain.
- Fact records preserve one row per flight.
- Business-rule violations are understood.
- Reliability scores remain within 0–100.
- ML splits and target distribution are suitable.
- Every expected year-month partition is present.
